# Calculate Defensive Features

## Import Functions

In [1]:
import numpy as np
import pandas as pd

In [ ]:
import sys
from pathlib import Path
# Add project root to path for imports
ROOT = Path.cwd().parent  
sys.path.append(str(ROOT))

from src.data_io.save_load import load_json, load_dataframe
from src.processing.indexing import build_tracking_time_index
from src.features.traj.compute import compute_defense_features_for_shots_refactored
from src.processing.tracking_cleaning import dedupe_tracking_events

ImportError: cannot import name 'compute_defense_features_for_shots_refactored' from 'src.features.defense.computing_features_updated' (/Users/cadenp/Documents/GitHub/DSC180B_FinalProject/src/features/defense/computing_features_updated.py)

## Load Data

In [ ]:
shots_df = load_dataframe("shots/all_season_shots")
# Load tracking data
TRACKING_JSON = ROOT / "data" / "processed" / "tracking" / "0021500622_processed.json"
tracking_events_raw = load_json(TRACKING_JSON)

In [4]:
# Flatten tracking events
tracking_rows = []

for event in tracking_events_raw:
    game_id = event["gameid"]
    event_id = event["event_id"]
    quarter = event["quarter"]

    for frame in event["frames"]:
        tracking_rows.append({
            "GAME_ID": game_id,
            "EVENT_ID": event_id,
            "PERIOD": quarter,
            "frame_id": frame["frame_id"],
            "game_clock": frame["game_clock"],
            "shot_clock": frame.get("shot_clock"),
            "players": frame["players"],   # keep raw for now
            "ball": frame.get("ball")
        })

tracking_df = pd.DataFrame(tracking_rows)

### Filter by specific Game
We will change this step when adding all the games

In [5]:
GAME_ID = tracking_df.loc[0, "GAME_ID"]

shots_g = shots_df[shots_df["GAME_ID"] == GAME_ID]
tracking_g = tracking_df[tracking_df["GAME_ID"] == GAME_ID]

In [6]:
tracking_g.head()

,GAME_ID,EVENT_ID,PERIOD,frame_id,game_clock,shot_clock,players,ball
0,21500622,1,1,1,623.59,15.99,"[{'teamid': 1610612739, 'playerid': 2544, 'x':...","{'x': 28.64068, 'y': 45.40327, 'z': 3.54135}"
1,21500622,1,1,2,623.55,15.96,"[{'teamid': 1610612739, 'playerid': 2544, 'x':...","{'x': 28.52319, 'y': 45.27989, 'z': 3.6859}"
2,21500622,1,1,3,623.51,15.93,"[{'teamid': 1610612739, 'playerid': 2544, 'x':...","{'x': 28.38205, 'y': 45.16353, 'z': 3.80584}"
3,21500622,1,1,4,623.47,15.90,"[{'teamid': 1610612739, 'playerid': 2544, 'x':...","{'x': 28.21808, 'y': 45.0552, 'z': 3.89686}"
4,21500622,1,1,5,623.43,15.87,"[{'teamid': 1610612739, 'playerid': 2544, 'x':...","{'x': 28.03211, 'y': 44.95588, 'z': 3.95463}"


In [7]:
shots_g.head()

,GAME_ID,SHOT_EVENT_ID,PERIOD,game_clock,PLAYER_ID,TEAM_ID,x_ft,y_ft,xFG_offense,xPPS_offense,SHOT_MADE_FLAG
104324,21500622,2,1,703,101106,1610612744,0.000000,0.083333,0.913826,1.827652,1
104325,21500622,3,1,676,2544,1610612739,-1.166667,0.666667,0.639951,1.279902,1
104326,21500622,4,1,663,201939,1610612744,-10.416667,21.166667,0.358939,1.076816,1
104327,21500622,5,1,640,202681,1610612739,-1.000000,1.333333,0.469154,0.938307,0
104328,21500622,7,1,628,202691,1610612744,-12.083333,10.083333,0.402172,0.804343,1


### Build Tracking Index

In [8]:
# Build tracking event index
# Dedupe the tracking events first to ensure we have a clean set of events for indexing
clean_tracking = dedupe_tracking_events(tracking_events_raw)
event_index_df = build_tracking_time_index(clean_tracking)
event_index_df.head()

,gameid,quarter,event_list_idx,gc_start,gc_end,gc_center,gc_span,n_frames_total,n_frames_gc,gc_monotone_frac
0,21500622,1,0,623.59,617.63,620.610,5.96,150,150,1.000000
1,21500622,1,1,623.59,610.24,616.915,13.35,335,335,1.000000
2,21500622,1,2,531.89,524.41,528.150,7.48,438,438,0.997712
3,21500622,1,3,530.37,509.41,519.890,20.96,525,525,1.000000
4,21500622,1,4,515.37,496.41,505.890,18.96,475,475,1.000000


### Match shots with tracking index

## Computer Defensive Features

In [9]:
defense_features_df = compute_defense_features_for_shots_refactored(
    shots_g=shots_g,
    tracking_events=clean_tracking,
    event_index=event_index_df
)

Attaching tracking events to shots (Vectorized)...
Error on shot 104361: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''
Defense features summary: {'no_event': 43, 'no_release': 8, 'feat_error': 0, 'exception': 2, 'ok': 108}


In [12]:
print(defense_features_df.shape)
print(defense_features_df.columns)
defense_features_df.head()

(108, 31)
Index(['GAME_ID', 'SHOT_EVENT_ID', 'tracking_event_id', 'release_frame_idx',
       'event_list_idx', 'PERIOD', 'game_clock', 'PLAYER_ID', 'TEAM_ID',
       'x_ft', 'y_ft', 'xFG_offense', 'xPPS_offense', 'SHOT_MADE_FLAG',
       'close_def_dist_release', 'closest_def_dist', 'close_def_id',
       'num_defenders_tracked', 'w0_close_def_dist_mean',
       'w0_close_def_dist_min', 'w0_shooter_speed_mean',
       'w0_shooter_speed_max', 'w0_def_speed_mean', 'w0_closing_speed_mean',
       'w1_close_def_dist_mean', 'w1_close_def_dist_min',
       'w1_shooter_speed_mean', 'w1_shooter_speed_max', 'w1_def_speed_mean',
       'w1_closing_speed_mean', 'shooter_speed'],
      dtype='object')


,GAME_ID,SHOT_EVENT_ID,tracking_event_id,release_frame_idx,event_list_idx,PERIOD,game_clock,PLAYER_ID,TEAM_ID,x_ft,...,w0_shooter_speed_max,w0_def_speed_mean,w0_closing_speed_mean,w1_close_def_dist_mean,w1_close_def_dist_min,w1_shooter_speed_mean,w1_shooter_speed_max,w1_def_speed_mean,w1_closing_speed_mean,shooter_speed
shot_index,,,,,,,,,,,,,,,,,,,,,
104329,21500622,8,1,149,0,1,617,201567,1610612739,-2.083333,...,17.254147,14.356321,-2.613488,7.333281,3.318036,9.505700,14.104240,13.449103,4.037829,15.265210
104330,21500622,10,2,334,1,1,610,202691,1610612744,14.000000,...,12.826330,6.755303,-2.369078,3.256696,1.215107,4.887627,8.311577,4.511752,0.124503,8.992652
104335,21500622,28,17,359,3,1,516,2544,1610612739,-4.333333,...,12.669068,13.593923,-0.954460,8.346007,3.311832,8.282123,11.600487,12.544824,4.389448,11.137788
104336,21500622,30,19,309,4,1,503,203110,1610612744,-11.250000,...,13.074742,9.044283,0.895768,4.037110,0.854171,2.731713,5.712955,6.338044,-4.468072,10.117841
104337,21500622,34,21,520,5,1,486,101106,1610612744,1.833333,...,12.732398,6.788480,5.756779,4.713687,1.554654,6.572920,8.257189,4.403070,-1.450470,10.063422


In [13]:
defense_features_df.to_parquet("../data/processed/def_features/defense_features.parquet", engine="pyarrow", index=False)